In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import numpy as np

from src.utils import (
    get_args,
    set_seed,
    get_datesets_and_loaders,
    get_trained_VAE,
    get_trained_VAE_with_domain_classifier,
    get_trained_classifier,
    get_trained_classifier_Base,
    test_model,
    prepare_report,
    run_all_senario
)
from src.tupl import run_tupl

/home/asad/workspace/DomainProject/changeDomain/notebooks/effective-gzsda/gzsda/src/utils.py:3: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.1)
  import scipy


In [3]:
DOMAIN_SET =['A','D','W']
DATA_DIR = './data/Office31/'
DATASET_DETAILS = {
    "prefix": 'office-',
    "suffix": '-resnet50-noft.mat',
    "resnet_feature": 'resnet50_features',
    "split_file_name": 'instanceSplit_office31_unseen15.mat',
}
NUM_LABELS=31

In [4]:
import json
from pathlib import Path

RESULT_OBJ_PATH = "./result/json/office31.json"
RESULT_CSV_PATH = "./result/csv/office31.csv"
path = Path(RESULT_OBJ_PATH)

if path.exists():
    with path.open("r", encoding="utf-8") as f:
        result = json.load(f)
else:
    result = {}

result.keys()

dict_keys(['base', 'CCVAE', 'our0', 'our_GRE', 'TUPL'])

In [5]:
base = "base"
CCVAE = "CCVAE"
our0 = "our0"
our_GRE = "our_GRE"
tupl = "TUPL"

# clear last result
# result.pop(base, None)
# result.pop(CCVAE, None)
# result.pop(our0, None)
# result.pop(our_GRE, None)
# result.pop(tupl, None)

## Base

In [6]:
def main_base(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    classifier = get_trained_classifier_Base(
        data_loaders=data_loaders,
        NUM_LABELS=NUM_LABELS,
        device=device)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [7]:
if base not in result:
    result[base] = run_all_senario(main_base, DOMAIN_SET)

## GZSDA

In [8]:
def main_gzsda(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE(
        data_loaders=data_loaders,
        args=args,
        device=device)

    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [9]:
if CCVAE not in result:
    result[CCVAE] = run_all_senario(main_gzsda, DOMAIN_SET)

## m0

In [10]:
def main_m0(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE(
        data_loaders=data_loaders,
        args=args,
        device=device)

    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device,
        change_policy_epoch=30)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [11]:
if our0 not in result:
    result[our0] = run_all_senario(main_m0, DOMAIN_SET)

## m1: seperate after encoder

In [12]:
def main_m1(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    datasets, data_loaders = get_datesets_and_loaders(
        args=args,
        DOMAIN_SET=DOMAIN_SET,
        DATA_DIR=DATA_DIR,
        DATASET_DETAILS=DATASET_DETAILS)

    vae = get_trained_VAE_with_domain_classifier(
        data_loaders=data_loaders,
        args=args,
        device=device)
        
    classifier = get_trained_classifier(
        data_loaders=data_loaders,
        vae=vae,
        NUM_LABELS=NUM_LABELS,
        device=device,
        change_policy_epoch=30)

    return test_model(classifier, datasets['test'], data_loaders['test'], device)

In [13]:
if our_GRE not in result:
    result[our_GRE] = run_all_senario(main_m1, DOMAIN_SET)

## TUPL

In [14]:
def main_tupl(args):
    set_seed(args)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    acc_s, acc_u, h = run_tupl(
        data_root="./data/",
        dataset="office31",
        source=args.sourceDomainIndex,
        target=args.targetDomainIndex,
        trial=args.trialIndex,
        seed=args.seed,
        device=device,
        quiet=True,
        return_model=False,
    )
    print('seen acc:{:2.4f}, unseen acc:{:2.4f}, H:{:2.4f}'.format(acc_s / 100, acc_u / 100, h / 100))
    return None, None, acc_s / 100.0, acc_u / 100.0

In [15]:
if tupl not in result:
    result[tupl] = run_all_senario(main_tupl, DOMAIN_SET)

## Merge results

In [16]:
with open(RESULT_OBJ_PATH, "w") as f:
    json.dump(result, f, indent=2)

In [17]:
# ignore our0
result.pop(our0, None)

{'A -> D': 'Seen:     87.26 ± 2.34\nUnseen:   88.32 ± 2.05\nH-mean:   87.63 ± 1.19',
 'A -> W': 'Seen:     85.32 ± 1.23\nUnseen:   82.57 ± 2.36\nH-mean:   83.79 ± 0.94',
 'D -> A': 'Seen:     80.97 ± 1.59\nUnseen:   63.79 ± 1.96\nH-mean:   71.18 ± 0.61',
 'D -> W': 'Seen:     94.06 ± 1.83\nUnseen:   96.06 ± 1.14\nH-mean:   94.96 ± 0.42',
 'W -> A': 'Seen:     80.56 ± 1.78\nUnseen:   62.61 ± 1.88\nH-mean:   70.28 ± 0.59',
 'W -> D': 'Seen:     97.66 ± 0.38\nUnseen:   99.76 ± 0.15\nH-mean:   98.70 ± 0.16'}

In [18]:
import pandas as pd
import re

rows = [(k, m, result[m][k]) for m in result for k in result[m]]
df = pd.DataFrame(rows, columns=['domain', 'method', 'values'])

def extract_metrics(text):
    matches = dict(re.findall(r'(\w+):\s+([\d.]+\s*±\s*[\d.]+)', text))
    return pd.Series(matches)

df[['seen', 'unseen', 'H-mean']] = df['values'].apply(extract_metrics)
df = df[['domain', 'method', 'seen', 'unseen', 'H-mean']]

# sort
df['method'] = pd.Categorical(df['method'], categories=[base, CCVAE, tupl, our0, our_GRE], ordered=True)
df = df.sort_values(['domain', 'method']).reset_index(drop=True)

df

,domain,method,seen,unseen,H-mean
0,A -> D,base,92.32 ± 1.37,67.06 ± 4.20,77.30 ± 2.48
1,A -> D,CCVAE,87.73 ± 2.36,87.48 ± 2.46,87.42 ± 1.37
2,A -> D,TUPL,72.92 ± 1.90,64.69 ± 3.11,68.43 ± 2.32
3,A -> D,our_GRE,85.60 ± 2.53,85.66 ± 2.33,85.44 ± 1.39
4,A -> W,base,91.83 ± 0.44,58.19 ± 3.21,71.07 ± 2.47
5,A -> W,CCVAE,87.54 ± 1.09,82.67 ± 1.95,84.95 ± 0.95
6,A -> W,TUPL,69.83 ± 2.15,59.52 ± 2.95,63.89 ± 1.21
7,A -> W,our_GRE,85.19 ± 1.30,83.15 ± 1.38,84.10 ± 0.81
8,D -> A,base,87.29 ± 1.17,38.12 ± 1.18,53.02 ± 1.17
9,D -> A,CCVAE,83.59 ± 1.60,58.52 ± 1.63,68.70 ± 0.66


In [19]:
df.to_csv(RESULT_CSV_PATH, index=False)